# 03a_train_sarsa – Train SARSA

**Chạy độc lập trên Kaggle.**

Flow:
1. Setup
2. Load data
3. Train SARSA (500 episodes trên train stream)
4. Save pkl → `experiments/saved_models/sarsa.pkl`
5. Learning curve

## 1. Setup

In [ ]:
import sys, os
import numpy as np
import pandas as pd

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from config import FULL_WINDOW_BATCHES, BATCH_SIZE
from utils.data_loader import AirlinesDataLoader
from models.LightGBM import LightGBM
from metrics.stream_metrics import StreamMetrics
from environment.drift_env import DriftStreamEnv
from utils.visualizer import plot_cumulative_reward

SAVE_DIR = os.path.join(PROJECT_ROOT, 'RL-concept-drift', 'experiments', 'saved_models')
print('Setup complete!')
from agents.td_agent import SARSAAgent

## 2. Load Data

In [ ]:
loader = AirlinesDataLoader()
X_train, y_train = loader.get_initial_train_data()

print(f'Train stream : {loader.n_train_batches} batches')
print(f'Test  stream : {loader.n_test_batches} batches')

## 3. Setup Environment

In [ ]:
clf     = LightGBM()
metrics = StreamMetrics()
env     = DriftStreamEnv(loader, clf, metrics)
print('Environment ready.')

## 4. Train SARSA

- Epsilon cố định = 0.1 (không decay) – môi trường non-stationary
- `mode='train'` → chỉ dùng train stream (70% đầu)
- Mỗi episode bắt đầu từ `TD_EPISODE_STARTS` khác nhau để tránh học vẹt

In [ ]:
N_EPISODES = 2000

agent = SARSAAgent()
print(f'Agent    : {agent.name}')
print(f'Epsilon  : {agent.epsilon} (fixed)')
print(f'Gamma    : {agent.gamma}')
print(f'Alpha    : {agent.alpha}')
print(f'Episodes : {N_EPISODES}')

history = agent.train(env, n_episodes=N_EPISODES, verbose=True)

## 5. Save Agent

In [ ]:
save_path = os.path.join(SAVE_DIR, 'sarsa.pkl')
agent.save(save_path)
print(f'Saved → {save_path}')
print(f'States visited : {len(agent.Q)}')

## 6. Learning Curve

In [ ]:
rewards = history['episode_rewards']
print(f'Avg reward (all)    : {np.mean(rewards):.4f}')
print(f'Avg reward (last 10): {np.mean(rewards[-10:]):.4f}')
print(f'Best episode reward : {max(rewards):.4f}')
print(f'Action distribution : {history["action_counts"]}')

plot_cumulative_reward(
    {'SARSA': rewards},
    title    = 'SARSA – Episode Reward over Training (Train Stream)',
    filename = 'sarsa_learning_curve.png',
)

## 7. Quick Sanity Check – Greedy Policy

In [ ]:
# Set epsilon = 0.0 → pure greedy để kiểm tra policy đã học
agent.epsilon = 0.0
print(f'Top 10 states by Q-value spread:')
items = sorted(
    agent.Q.items(),
    key=lambda x: np.max(x[1]) - np.min(x[1]),
    reverse=True
)[:10]
for state, q in items:
    best_action = int(np.argmax(q))
    from environment.drift_env import ACTIONS
    print(f'  {state} → {ACTIONS[best_action]:<16} Q={np.round(q,3)}')

# Restore epsilon
agent.epsilon = 0.1